# Incremental personalization

For each of the 23 users, the segments are processed in chronological order under a test-then-train protocol. Three methods are compared on a common test set:

- Static base model $B_0$: XGBoost trained on the other 22 users (leave-one-subject-out) and never updated.
- Incremental personalization: after the FMS report of segment $j$, $B_0$ is reloaded and $m$ additional
  boosting rounds are fitted on the target user's labelled windows of segments $1..j$ with time-decay weights.
- Rapid calibration (upper bound): also uses the labels of the first half of the windows of the
  current segment and is tested on the second half.

Uses `../data/window_features.csv`. `../data/frame_features.csv` is only needed to time the feature extraction.

In [ ]:
import os
import platform
import subprocess
import time
import warnings
from collections import OrderedDict

import imblearn
import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

DATA_PATH = os.path.join("..", "data", "window_features.csv")
FRAME_DATA_PATH = os.path.join("..", "data", "frame_features.csv")
OUT_DIR = "results"
os.makedirs(OUT_DIR, exist_ok=True)
PRED_PATH = os.path.join(OUT_DIR, "predictions_per_window.csv")
TIME_PATH = os.path.join(OUT_DIR, "timing_per_update.csv")

SEED = 42
np.random.seed(SEED)
NUM_CLASS = 4
LABELS = [0, 1, 2, 3]
CLASS_NAMES = ["None", "Low", "Medium", "High"]
NTHREAD = 2
XGB_PARAMS = {"objective": "multi:softprob", "num_class": NUM_CLASS, "max_depth": 6, "eta": 0.05,
              "subsample": 0.8, "colsample_bytree": 0.8, "gamma": 0.1, "alpha": 0.1, "lambda": 1.0,
              "eval_metric": "mlogloss", "seed": SEED, "nthread": NTHREAD}
NUM_ROUND_BASE = 400            # boosting rounds of B0 (1600 trees for four classes)
WARM_START_ROUNDS = 200         # m: additional boosting rounds per update
DECAY_HALF_LIFE_SEG = 10        # H: half-life of the time-decay weights, in segments
CAL_RATIO = 0.5                 # rapid calibration: the first floor(n * 0.5) windows of a segment are labelled
PRESENT_MIN_WINDOWS = 10        # per-user metrics use classes with at least 10 windows for that user
RESUME = True                   # skip users whose predictions are already saved

USER_COL, SCENARIO_COL, SEGMENT_COL, LABEL_COL, WIN_COL = "participant", "simulation", "segment_marker", "cs", "seg_win_index"
feature_cols_raw = [
    "LeftPupilPosInSensorX", "LeftPupilPosInSensorY", "RightPupilPosInSensorX", "RightPupilPosInSensorY",
    "GazeOriginLclSpc_X", "GazeOriginLclSpc_Y", "GazeOriginLclSpc_Z",
    "GazeOriginWrldSpc_X", "GazeOriginWrldSpc_Y", "GazeOriginWrldSpc_Z",
    "NrmLeftEyeOriginX", "NrmLeftEyeOriginY", "NrmLeftEyeOriginZ",
    "NrmRightEyeOriginX", "NrmRightEyeOriginY", "NrmRightEyeOriginZ",
    "HeadQRotationX", "HeadQRotationY", "HeadQRotationZ", "HeadQRotationW",
    "HeadEulX", "HeadEulY", "HeadEulZ",
]
METHODS = OrderedDict([("static", "Static base model B0"), ("main", "Incremental personalization"),
                       ("rapid_cal", "Rapid calibration (upper bound)")])


def _cpu_name():
    try:
        out = subprocess.run(["powershell", "-NoProfile", "-Command",
                              "(Get-CimInstance Win32_Processor | Select-Object -First 1).Name"],
                             capture_output=True, text=True, timeout=30)
        return out.stdout.strip() or platform.processor()
    except Exception:
        return platform.processor()


ENV_INFO = OrderedDict([("CPU", _cpu_name()), ("LogicalCPUs", os.cpu_count()), ("XGBoost_nthread", NTHREAD),
                        ("OS", platform.platform()), ("Python", platform.python_version()),
                        ("xgboost", xgb.__version__), ("scikit-learn", sklearn.__version__),
                        ("imbalanced-learn", imblearn.__version__), ("pandas", pd.__version__),
                        ("numpy", np.__version__), ("m", WARM_START_ROUNDS), ("H", DECAY_HALF_LIFE_SEG),
                        ("CAL_RATIO", CAL_RATIO)])
pd.Series(ENV_INFO).to_csv(os.path.join(OUT_DIR, "environment.csv"), header=["value"])

## Data and chronological segment table

In [ ]:
def map_features(df):
    return [f"{c}__mean" if f"{c}__mean" in df.columns else c for c in feature_cols_raw
            if f"{c}__mean" in df.columns or c in df.columns]


def clean_df(target_df, feat_cols):
    needed = feat_cols + [LABEL_COL, SEGMENT_COL, SCENARIO_COL, USER_COL]
    tdf = target_df.dropna(subset=needed).copy()
    tdf[LABEL_COL] = tdf[LABEL_COL].astype(int)
    return tdf[tdf[LABEL_COL].isin(LABELS)]


df = pd.read_csv(DATA_PATH).dropna(how="all")
feature_cols = map_features(df)
df = clean_df(df, feature_cols)
df[USER_COL] = df[USER_COL].astype(int)

# segment_marker increases with time within each user; windows are ordered within each segment.
df = df.sort_values([USER_COL, SEGMENT_COL, WIN_COL]).reset_index(drop=True)

seg = (df.groupby([USER_COL, SEGMENT_COL], sort=True)
         .agg(scenario=(SCENARIO_COL, "first"), label=(LABEL_COL, "first"), n_win=(LABEL_COL, "size"))
         .reset_index()
         .sort_values([USER_COL, SEGMENT_COL])
         .reset_index(drop=True))
seg["seg_idx"] = seg.groupby(USER_COL).cumcount()

cnt = pd.crosstab(df[USER_COL], df[LABEL_COL]).reindex(columns=LABELS, fill_value=0)
present_classes = {u: [c for c in LABELS if cnt.loc[u, c] >= PRESENT_MIN_WINDOWS] for u in cnt.index}

## Model functions

In [ ]:
def ros_resample(X, y):
    # also returns the indices, so the sample weights follow the copies
    if len(np.unique(y)) >= 2:
        ros = RandomOverSampler(random_state=SEED)
        Xr, yr = ros.fit_resample(X, y)
        return Xr, yr, ros.sample_indices_
    return X, y, np.arange(len(y))


def train_from_scratch(X, y):
    t_all = time.perf_counter()
    Xr, yr, _ = ros_resample(X, y)
    scaler = StandardScaler()
    dtrain = xgb.DMatrix(scaler.fit_transform(Xr), label=yr)
    t0 = time.perf_counter()
    bst = xgb.train(XGB_PARAMS, dtrain, num_boost_round=NUM_ROUND_BASE)
    return bst, scaler, time.perf_counter() - t0, time.perf_counter() - t_all


def warm_start(B0_raw, S0, X, y, w):
    # B0 is reloaded at every update, so each update starts from the same model
    t_all = time.perf_counter()
    Xr, yr, idx = ros_resample(X, y)
    dtrain = xgb.DMatrix(S0.transform(Xr), label=yr, weight=np.asarray(w, float)[idx])
    t0 = time.perf_counter()
    bst = xgb.train(XGB_PARAMS, dtrain, num_boost_round=WARM_START_ROUNDS, xgb_model=xgb.Booster(model_file=B0_raw))
    return bst, time.perf_counter() - t0, time.perf_counter() - t_all


def predict_timed(bst, scaler, X):
    d = xgb.DMatrix(scaler.transform(X))
    t0 = time.perf_counter()
    prob = bst.predict(d)
    return np.argmax(prob, axis=1), (time.perf_counter() - t0) / len(X) * 1000.0


def single_window_latency_ms(bst, scaler, x_row):
    # timed part: standardization, input matrix and prediction for one window
    t0 = time.perf_counter()
    bst.predict(xgb.DMatrix(scaler.transform(x_row.reshape(1, -1))))
    return (time.perf_counter() - t0) * 1000.0


def decay_weights(ages):
    # w = 2^(-age / H), age in segments, normalized to mean 1
    w = 0.5 ** (np.asarray(ages, float) / DECAY_HALF_LIFE_SEG)
    return w / w.mean()


def split_segment(idx_t):
    # first floor(n * CAL_RATIO) windows for calibration, the rest for testing
    n = len(idx_t)
    n_cal = int(np.floor(n * CAL_RATIO))
    if n_cal == 0:
        return np.array([], int), np.array([], int)
    cal, rest = idx_t[:n_cal], idx_t[n_cal:]
    last_cal_end = df.loc[cal, "seg_win_end_row"].max()
    test = np.array([i for i in rest if df.loc[i, "seg_win_start_row"] > last_cal_end], int)
    return cal, test

## Test-then-train loop

In [ ]:
users = sorted(df[USER_COL].unique())
done = (set(pd.read_csv(PRED_PATH, usecols=[USER_COL])[USER_COL].unique())
        if (RESUME and os.path.exists(PRED_PATH)) else set())
X_all = df[feature_cols].values
y_all = df[LABEL_COL].values.astype(int)
user_arr = df[USER_COL].values
seg_arr = df[SEGMENT_COL].values

for u in users:
    if u in done:
        continue
    m0 = user_arr != u
    B0, S0, t_tr0, t_all0 = train_from_scratch(X_all[m0], y_all[m0])
    B0_raw = B0.save_raw()
    time_rows = [dict(user=u, seg_idx=-1, method="B0_offline", train_s=t_tr0, update_total_s=t_all0,
                      n_trees=B0.num_boosted_rounds())]
    su = seg[seg[USER_COL] == u].reset_index(drop=True)
    rows = [np.flatnonzero((user_arr == u) & (seg_arr == s)) for s in su[SEGMENT_COL]]
    model_main = B0
    out_rows = []

    for j in range(len(su)):
        idx_t = rows[j]
        Xt = X_all[idx_t]
        cal, test = split_segment(idx_t)
        pred = {m: np.full(len(idx_t), np.nan) for m in METHODS}

        # Static base model B0
        p, inf = predict_timed(B0, S0, Xt)
        pred["static"][:] = p
        time_rows.append(dict(user=u, seg_idx=j, method="static", infer_ms_per_window=inf,
                              latency_single_ms=single_window_latency_ms(B0, S0, Xt[0]),
                              n_trees=B0.num_boosted_rounds()))

        # Incremental personalization: the model updated with segments 1..j-1 detects segment j
        p, inf = predict_timed(model_main, S0, Xt)
        pred["main"][:] = p
        time_rows.append(dict(user=u, seg_idx=j, method="main_predict", infer_ms_per_window=inf,
                              latency_single_ms=single_window_latency_ms(model_main, S0, Xt[0]),
                              n_trees=model_main.num_boosted_rounds()))

        # Rapid calibration: segments 1..j-1 plus the calibration windows of segment j
        if len(test) > 0:
            ii = np.concatenate(rows[:j] + [cal])
            ages = np.concatenate([np.repeat(j - su.loc[k, "seg_idx"], len(rows[k])) for k in range(j)]
                                  + [np.zeros(len(cal))])
            bst_c, t_tr, t_tot = warm_start(B0_raw, S0, X_all[ii], y_all[ii], decay_weights(ages))
            p, inf = predict_timed(bst_c, S0, X_all[test])
            pred["rapid_cal"][np.searchsorted(idx_t, test)] = p
            time_rows.append(dict(user=u, seg_idx=j, method="rapid_cal_update", n_user_windows=len(ii),
                                  n_cal=len(cal), train_s=t_tr, update_total_s=t_tot, infer_ms_per_window=inf,
                                  latency_single_ms=single_window_latency_ms(bst_c, S0, X_all[test[0]]),
                                  n_trees=bst_c.num_boosted_rounds()))

        # After the FMS report of segment j: update with segments 1..j
        ii = np.concatenate(rows[:j + 1])
        ages = np.concatenate([np.repeat(j - su.loc[k, "seg_idx"], len(rows[k])) for k in range(j + 1)])
        model_main, t_tr, t_tot = warm_start(B0_raw, S0, X_all[ii], y_all[ii], decay_weights(ages))
        time_rows.append(dict(user=u, seg_idx=j, method="main_update", n_user_windows=len(ii),
                              train_s=t_tr, update_total_s=t_tot, n_trees=model_main.num_boosted_rounds()))

        test_set, cal_set = set(test.tolist()), set(cal.tolist())
        r = su.loc[j]
        for k, irow in enumerate(idx_t):
            row = dict(participant=u, scenario=r["scenario"], segment_marker=r[SEGMENT_COL], seg_idx=j,
                       win_idx=int(df.loc[irow, WIN_COL]), n_win_seg=len(idx_t),
                       role="test" if irow in test_set else ("cal" if irow in cal_set else "none"),
                       y_true=int(y_all[irow]))
            for m in METHODS:
                row[f"pred_{m}"] = pred[m][k]
            out_rows.append(row)

    pd.DataFrame(out_rows).to_csv(PRED_PATH, mode="a", header=not os.path.exists(PRED_PATH), index=False)
    pd.DataFrame(time_rows).to_csv(TIME_PATH, mode="a", header=not os.path.exists(TIME_PATH), index=False)

## Pooled metrics on the common test set

In [ ]:
pred = pd.read_csv(PRED_PATH)
test = pred[pred.role == "test"]

pooled = []
for m, name in METHODS.items():
    y, p = test["y_true"].astype(int), test[f"pred_{m}"].astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, labels=LABELS, average=None, zero_division=0)
    row = OrderedDict(method=name, Accuracy=accuracy_score(y, p),
                      MacroF1=f1_score(y, p, labels=LABELS, average="macro", zero_division=0))
    for metric, arr in [("Precision", pr), ("Recall", rc), ("F1", f1)]:
        for c in LABELS:
            row[f"{metric}_{CLASS_NAMES[c]}"] = arr[c]
    pooled.append(row)
pooled = pd.DataFrame(pooled)
pooled.to_csv(os.path.join(OUT_DIR, "pooled_metrics_common_test_set.csv"), index=False)
print(pooled.round(3).T.to_string())

## Row-normalized confusion matrices

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
                     "mathtext.fontset": "stix"})
titles = {"static": "Static base model $B_0$", "main": "Incremental personalization",
          "rapid_cal": "Rapid calibration (upper bound)"}
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.75))
for k, (ax, m) in enumerate(zip(axes, METHODS)):
    M = confusion_matrix(test["y_true"].astype(int), test[f"pred_{m}"].astype(int), labels=LABELS)
    R = M / M.sum(axis=1, keepdims=True)
    ax.imshow(R, cmap="Blues", vmin=0, vmax=1)
    for i in range(4):
        for jj in range(4):
            color = "white" if R[i, jj] > 0.55 else "#1A1A1A"
            ax.text(jj, i - 0.13, f"{R[i, jj]:.2f}", ha="center", va="center", fontsize=11, color=color)
            ax.text(jj, i + 0.25, f"({M[i, jj]})", ha="center", va="center", fontsize=8.5,
                    color="white" if R[i, jj] > 0.55 else "#444444")
    ax.set_xticks(range(4)); ax.set_xticklabels(CLASS_NAMES, fontsize=9.5)
    ax.set_yticks(range(4)); ax.set_yticklabels(CLASS_NAMES if k == 0 else [], fontsize=9.5)
    ax.set_xlabel("Predicted class", fontsize=10.5)
    ax.set_title(titles[m], fontsize=11, pad=5)
axes[0].set_ylabel("True class", fontsize=10.5)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "confusion_matrices.pdf"), bbox_inches="tight")
plt.show()

## Computation time
Detection time = feature extraction from the 60 raw frames of a window + standardization, building the input
matrix and prediction for that window. Its standard deviation combines the two parts as independent terms.

In [ ]:
def exp_smooth_skip_nan(arr, alpha=0.08):
    # same smoothing as in preprocessing (factor 0.08)
    out = np.empty_like(arr, dtype=float)
    prev = np.nan
    for i, x in enumerate(arr):
        if np.isnan(x):
            out[i] = np.nan
            prev = np.nan
        else:
            prev = x if np.isnan(prev) else alpha * x + (1 - alpha) * prev
            out[i] = prev
    return out


frames = pd.read_csv(FRAME_DATA_PATH, usecols=feature_cols_raw + ["window_marker"], nrows=60000)
mu, sd = frames[feature_cols_raw].mean().values, frames[feature_cols_raw].std().values
chunks = []
for _, g in frames.dropna(subset=["window_marker"]).groupby("window_marker", sort=False):
    v = g[feature_cols_raw].values
    for s0 in range(0, len(v) - 59, 60):
        chunks.append(v[s0:s0 + 60])
    if len(chunks) >= 500:
        break
fe_ms = []
for c in chunks:
    t0 = time.perf_counter()
    keep = (np.abs((c - mu) / sd) <= 3).all(axis=1)
    cs_ = c[keep] if keep.any() else c
    sm = np.column_stack([exp_smooth_skip_nan(cs_[:, k]) for k in range(cs_.shape[1])])
    _ = np.nanmean(sm, axis=0)
    fe_ms.append((time.perf_counter() - t0) * 1000.0)
fe_ms = np.array(fe_ms)

tm = pd.read_csv(TIME_PATH)
TIME_MAP = {"static": ("static", None), "main": ("main_predict", "main_update"),
            "rapid_cal": ("rapid_cal_update", "rapid_cal_update")}
comp_time = []
for m, (pred_key, upd_key) in TIME_MAP.items():
    pr = tm[tm.method == pred_key]
    up = tm[tm.method == upd_key] if upd_key else tm.iloc[0:0]
    lat = pr["latency_single_ms"]
    comp_time.append(OrderedDict(
        method=METHODS[m],
        boosting_rounds=int(pr["n_trees"].max()),
        update_time_s_mean=up["update_total_s"].mean() if len(up) else np.nan,
        update_time_s_sd=up["update_total_s"].std(ddof=1) if len(up) > 1 else np.nan,
        update_time_s_max=up["update_total_s"].max() if len(up) else np.nan,
        inference_ms_per_window=pr["infer_ms_per_window"].mean(),
        detection_ms_mean=lat.mean() + fe_ms.mean(),
        detection_ms_sd=np.sqrt(lat.std(ddof=1) ** 2 + fe_ms.std(ddof=1) ** 2),
    ))
comp_time = pd.DataFrame(comp_time)
b0 = tm[tm.method == "B0_offline"]
print(f"feature extraction: {fe_ms.mean():.2f} +/- {fe_ms.std(ddof=1):.2f} ms per window")
print(f"B0 offline training: {b0['train_s'].mean():.2f} +/- {b0['train_s'].std(ddof=1):.2f} s")
comp_time.to_csv(os.path.join(OUT_DIR, "computation_time.csv"), index=False)
print(comp_time.round(3).to_string(index=False))